In [ ]:
def standardize_element_names(df):
    """Normalize chemical headers to canonical standard names."""
    renames = {}
    for col in df.columns:
        clean_col = re.sub(r'[^a-zA-Z0-9%_]', '', str(col)).upper()
        if clean_col in ELEMENT_ALIAS_MAP:
            renames[col] = ELEMENT_ALIAS_MAP[clean_col]
        else:
            for key, std in ELEMENT_ALIAS_MAP.items():
                if key in clean_col:
                    renames[col] = std
                    break
    return df.rename(columns=renames)


def clean_dataframe(df):
    """Universal cleaning routine for geochemical datasets."""
    if df is None or df.empty:
        return df

    df.columns = [str(c).strip() for c in df.columns]
    df = df.loc[:, ~df.columns.duplicated()]
    df = standardize_element_names(df)

    missing_vals = ['', 'nan', 'NaN', 'null', 'None', 'NA', '-', '--', 'n/a', '?']
    df = df.replace(missing_vals, np.nan)

    text_cols = ['sample', 'id', 'point', 'lithology', 'comments', 'unit', 'rock_type']
    for col in df.columns:
        is_text = any(t in col.lower() for t in text_cols)
        if not is_text:
            cleaned_series = df[col].astype(str).str.replace(r'^[<>]', '', regex=True)
            cleaned_series = pd.to_numeric(cleaned_series.str.replace(',', '.'), errors='coerce')
            if cleaned_series.notna().sum() > 0:
                df[col] = cleaned_series

    df = df.dropna(how='all')
    return df


def read_any_file(path):
    """Read any tabular file format with auto-detection of encoding/separator."""
    ext = Path(path).suffix.lower()
    if ext in ['.xlsx', '.xls', '.xlsm']:
        return pd.read_excel(path)

    encodings = ["utf-8", "cp1252", "latin-1", "iso-8859-1", "utf-16"]
    seps = [";", ",", "\t", "|"]

    for enc in encodings:
        for sep in seps:
            try:
                df = pd.read_csv(path, sep=sep, encoding=enc, engine="python", on_bad_lines="skip")
                if df.shape[1] > 1:
                    return df
            except Exception:
                pass

    try:
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            txt = f.read(3000)
        sep = max([";", ",", "\t", "|"], key=lambda d: txt.count(d))
        return pd.read_csv(path, sep=sep, engine="python", on_bad_lines="skip")
    except Exception:
        return pd.read_csv(path, engine="python", on_bad_lines="skip")

In [ ]:
import re
from pathlib import Path
from typing import Optional, Union

In [6]:
import pandas as pd
import numpy as np
df = pd.read_csv('working esa files/Major elements all.xlsx', encoding = 'cp1252')
print(df)

FileNotFoundError: [Errno 2] No such file or directory: 'working esa files/Major elements all.xlsx'

In [9]:
df = pd.read_excel('Major elements all.xlsx')
print(df)

                Sample   SiO2  Al2O3   MgO   CaO  Na2O   K2O  Fe2O3    MnO  \
0      MÁF - ALMOU 2MS  73.89  12.70  0.19  0.31  3.29  4.50   0.88  0.007   
1    MÁF - ALMOU 3MS-A  75.40  13.44  0.21  0.25  3.25  4.27   1.11  0.006   
2    MÁF - ALMOU 3MS-B  76.20  13.01  0.26  0.28  3.52  4.54   0.92  0.009   
3      MÁF - ALMOU 4MS  75.65  13.05  0.23  0.26  3.63  4.57   1.02  0.012   
4     MÁF - ALMOU KR-1  77.11  14.31  0.88  0.79  0.14  0.52   1.80  0.024   
5     MÁF - ALMOU KR-2  78.79  13.16  0.44  0.28  0.12  6.35   1.16  0.008   
6     MÁF - ALMOU PT-1  76.77  12.49  0.10  0.61  3.18  4.87   1.26  0.027   
7   MÁF - ALMOU PT-5-1  77.46  12.79  0.76  1.23  0.21  5.40   1.31  0.042   
8   MÁF - ALMOU PT-5-2  76.11  11.51  0.54  1.13  0.15  5.74   1.19  0.008   
9   MÁF - ALMOU PT-7-1  76.69  12.08  0.42  2.43  0.17  4.63   1.31  0.042   
10  MÁF - ALMOU PT-7-2  77.74  11.90  0.49  2.11  0.18  4.80   1.26  0.008   
11  MÁF - ALMOU PT-7-3  77.73  11.69  0.58  2.23  0.19  4.35   1

In [12]:
df = df.drop('Sample', axis =1)

In [13]:
print(df)

     SiO2  Al2O3   MgO   CaO  Na2O   K2O  Fe2O3    MnO   TiO2
0   73.89  12.70  0.19  0.31  3.29  4.50   0.88  0.007  0.086
1   75.40  13.44  0.21  0.25  3.25  4.27   1.11  0.006  0.092
2   76.20  13.01  0.26  0.28  3.52  4.54   0.92  0.009  0.110
3   75.65  13.05  0.23  0.26  3.63  4.57   1.02  0.012  0.090
4   77.11  14.31  0.88  0.79  0.14  0.52   1.80  0.024  0.105
5   78.79  13.16  0.44  0.28  0.12  6.35   1.16  0.008  0.087
6   76.77  12.49  0.10  0.61  3.18  4.87   1.26  0.027  0.085
7   77.46  12.79  0.76  1.23  0.21  5.40   1.31  0.042  0.083
8   76.11  11.51  0.54  1.13  0.15  5.74   1.19  0.008  0.080
9   76.69  12.08  0.42  2.43  0.17  4.63   1.31  0.042  0.085
10  77.74  11.90  0.49  2.11  0.18  4.80   1.26  0.008  0.085
11  77.73  11.69  0.58  2.23  0.19  4.35   1.16  0.033  0.087
12  64.60  16.50  0.55  0.96  0.13  2.09   1.69  0.018  0.240
13  71.50  12.40  0.53  0.84  0.13  1.60   1.40  0.017  0.220
14  69.30  11.00  0.68  0.77  0.09  1.38   1.83  0.015  0.190
15  62.4